# Unidad 09. Archivos, rutas y formatos de datos

**Duración orientativa:** 7 horas · **Versión:** 1.0 · **Unidad temática:** UT2

Los programas de datos necesitan leer información, transformarla y guardar resultados. Esta unidad muestra cómo trabajar con archivos de forma portable y segura, y cómo interpretar formatos de texto, CSV y JSON sin depender de la configuración concreta de un equipo.

## Resultados de aprendizaje

Al finalizar podrás construir rutas con pathlib; distinguir rutas absolutas y relativas; leer y escribir texto con codificación UTF-8; usar context managers para cerrar archivos; interpretar CSV con el módulo csv; serializar y leer JSON con json; y validar los datos y rutas antes de procesarlos.

## 1. Archivos y rutas portables

Un archivo tiene un contenido y una ubicación. La ubicación se expresa mediante una ruta. Una ruta absoluta describe una localización completa en un equipo concreto; una ruta relativa se interpreta desde la carpeta de trabajo actual. Las rutas absolutas de tu ordenador no sirven para distribuir un cuaderno a otra persona, por lo que este curso trabaja con rutas relativas a la raíz del proyecto.

Pathlib proporciona el objeto Path para construir rutas de forma compatible con Windows, macOS y Linux. En lugar de unir textos con barras manualmente, se usa el operador barra: carpeta / nombre_archivo. Path se encarga de usar el separador correcto del sistema. Antes de abrir un archivo, comprueba si existe y si es un archivo normal cuando el contexto lo requiera.

In [8]:
from pathlib import Path

raiz = Path.cwd().parent
ruta_datos = raiz / "datos" / "09_lectura_ejemplo.txt"

print("Carpeta actual:", raiz)
print("Ruta construida:", ruta_datos)
print("¿Existe?", ruta_datos.exists())
print("¿Es archivo?", ruta_datos.is_file())


Carpeta actual: C:\Users\pajim\OneDrive\Escritorio\Master\Programacion IA\Proyecto
Ruta construida: C:\Users\pajim\OneDrive\Escritorio\Master\Programacion IA\Proyecto\datos\09_lectura_ejemplo.txt
¿Existe? True
¿Es archivo? True


## 2. Texto y codificación

Un archivo de texto almacena caracteres codificados como bytes. UTF-8 es la codificación recomendada para este curso porque representa texto internacional y es el estándar más usado en la web y en muchos sistemas de datos. Indicar encoding igual a utf-8 de manera explícita evita que el resultado dependa de la configuración local del sistema.

Path ofrece read_text y write_text para casos sencillos. También puede utilizarse open dentro de una instrucción with. With es un context manager: abre el archivo, permite trabajar con él dentro del bloque y garantiza su cierre incluso si se produce un error. Cerrar recursos es importante porque libera el archivo y asegura que las operaciones pendientes se completen.

In [12]:
from pathlib import Path

# ruta = Path("datos") / "09_lectura_ejemplo.txt"
ruta = Path.cwd().parent / "datos" / "09_lectura_ejemplo.txt"
if ruta.is_file():
    with ruta.open(mode="r", encoding="utf-8") as archivo:
        contenido = archivo.read() # Sigue disponible al salir del bloque with?
    print("Caracteres leídos:", len(contenido))
    print(repr(contenido[:80]))

else:
    print("No se ha encontrado el archivo de ejemplo.")


Caracteres leídos: 69
'Lectura de ejemplo para la Unidad 09\nAula: A-01\nTemperatura: 21.4 °C\n'


### Escribir sin sobrescribir por accidente

Abrir un archivo en modo w crea el archivo si no existe y sustituye por completo su contenido si ya existe. El modo a añade al final. Antes de escribir, decide conscientemente si quieres crear, reemplazar o ampliar. Para trabajos de alumnado, usa un nombre propio en una carpeta de entregas y evita modificar recursos originales del curso.

El ejemplo siguiente usa un directorio temporal para que el cuaderno sea reproducible y no deje archivos de prueba en la carpeta del curso. En un proyecto real puedes sustituir esa ruta temporal por una ruta relativa preparada para la entrega.

In [15]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as carpeta_temporal:
    ruta_salida = Path(carpeta_temporal) / "resumen.txt"
    ruta_salida.write_text("Informe de prueba\nLecturas procesadas: 3\n", encoding="utf-8")
    print(ruta_salida.read_text(encoding="utf-8"))


Informe de prueba
Lecturas procesadas: 3



## 3. Datos tabulares en CSV

CSV significa valores separados por comas. Es un formato de texto común para tablas, pero no define por sí solo todo el contexto: el separador puede ser coma, punto y coma u otro; los valores pueden incluir comillas; y los números se leen inicialmente como texto. No es seguro procesar un CSV complejo con split. El módulo csv entiende separadores y comillas de forma adecuada.

DictReader interpreta la primera fila como encabezados y produce cada fila como un diccionario. Es cómodo porque permite acceder a los campos por su nombre. Convierte de forma explícita cada valor al tipo que representa y comprueba que los encabezados y datos necesarios existan.

In [14]:
import csv
from io import StringIO

texto_csv = "aula;temperatura;humedad\nA-01;21.4;45\nA-02;22.0;48\n"
archivo_csv = StringIO(texto_csv)
lector = csv.DictReader(archivo_csv, delimiter=";")

for fila in lector:
    temperatura = float(fila["temperatura"]) # Que tipo de dato es fila["temperatura"] antes de llamar a float
    humedad = int(fila["humedad"])
    print(fila["aula"], temperatura, humedad)


A-01 21.4 45
A-02 22.0 48


## 4. Datos estructurados en JSON

JSON es un formato de texto para intercambiar datos estructurados. Sus objetos se corresponden con diccionarios de Python, sus arrays con listas, sus cadenas con texto y sus valores true, false y null con True, False y None. El módulo json convierte entre JSON y estructuras básicas de Python.

Json dump escribe una estructura de Python en un archivo o flujo; json load la recupera. Para texto ya disponible en memoria se usan dumps y loads. El parámetro ensure_ascii igual a falso permite conservar caracteres como tildes en la representación, y indent mejora la legibilidad para revisión humana. JSON no puede guardar directamente cualquier objeto de Python; es una ventaja porque obliga a usar una representación simple e interoperable. No uses pickle para datos externos no confiables: puede ejecutar código al cargarlo.

In [ ]:
import json

lectura = {"aula": "A-01", "temperatura": 21.4, "sensor_activo": True, "observacion": None}
texto_json = json.dumps(lectura, ensure_ascii=False, indent=2)
lectura_recuperada = json.loads(texto_json)

print(texto_json)
print(lectura_recuperada["aula"])
print(type(lectura_recuperada))


## 5. Validar antes de procesar

Un programa que lee archivos debe comprobar sus supuestos: que la ruta existe, que se trata del tipo de archivo esperado, que la codificación es conocida, que los campos necesarios aparecen y que las conversiones de tipo tienen sentido. Una extensión como .csv o .json es una pista, no una garantía sobre el contenido.

Durante el desarrollo, separa tres pasos: localizar el archivo, leer su contenido y transformar los datos. Si aparece un error, esta separación permite saber si el problema está en la ruta, el formato o una transformación. No ocultes errores de lectura devolviendo resultados inventados: informa qué falta y conserva el contexto para revisarlo.

## Práctica guiada

1. Ejecuta el ejemplo de rutas y verifica que su resultado no contiene una ruta fija de otro ordenador.
2. Lee el archivo de texto y muestra con repr los últimos 40 caracteres.
3. Cambia temporalmente el nombre del archivo por uno inexistente y observa qué rama de la decisión se ejecuta. Restaura después el nombre.
4. Añade una fila al texto CSV y convierte correctamente sus dos medidas.
5. Añade un campo nuevo al diccionario JSON y comprueba que se recupera con el tipo esperado.

## Ejercicios

**Ejercicio 09.01 · Ruta portable.** Construye con Path la ruta relativa datos / 09_lectura_ejemplo.txt. Muestra si existe y si es archivo. Explica por qué no debes escribir una ruta absoluta de tu carpeta personal en un cuaderno que se distribuirá.

**Ejercicio 09.02 · Escribir y comprobar texto.** Usa un directorio temporal para crear un archivo llamado notas.txt con dos líneas UTF-8 que contengan una tilde. Léelo de nuevo, muestra repr del contenido y comprueba si coincide exactamente con el texto que escribiste.

**Ejercicio 09.03 · CSV con encabezados.** A partir del texto CSV aula;temperatura con dos filas, usa DictReader con separador punto y coma para construir una lista de temperaturas de tipo float. Muestra la lista y explica por qué no se puede sumar directamente la cadena leída.

**Ejercicio 09.04 · JSON legible.** Crea un diccionario con aula, ocupacion y proyector. Conviértelo a texto JSON con indent y caracteres no ASCII conservados; recupéralo; y comprueba el tipo de cada valor.

In [16]:
from pathlib import Path

ruta = Path.cwd().parent / "datos" / "09_lectura_ejemplo.txt"

if ruta:
    print("Si existe.")
    if ruta.is_file():
        print("Es un archivo.")


Si existe.
Es un archivo.


No se debe escribir dado que la ruta relativa no va a ser la misma en otro PC

Ejercicio 09.02 · Escribir y comprobar texto. Usa un directorio temporal para crear un archivo llamado notas.txt con dos líneas UTF-8 que contengan una tilde. Léelo de nuevo, muestra repr del contenido y comprueba si coincide exactamente con el texto que escribiste.

In [7]:
from tempfile import TemporaryDirectory
from pathlib import Path


texto_original = "Están comiendo\nEllos lo están pasando bien"

with TemporaryDirectory() as carpeta_temporal:
    ruta = Path(carpeta_temporal) / "notas.txt"
    with open(ruta, mode = "w", encoding = "utf-8") as archivo:
        archivo.write("Están comiendo")
        archivo.write("\nEllos lo están pasando bien")

    with open(ruta, mode = "r", encoding="utf-8") as archivo:
        contenido = archivo.read()
        print(repr(contenido))
        print(contenido == texto_original)


'Están comiendo\nEllos lo están pasando bien'
True


Ejercicio 09.03 · CSV con encabezados. A partir del texto CSV aula;temperatura con dos filas, usa DictReader con separador punto y coma para construir una lista de temperaturas de tipo float. Muestra la lista y explica por qué no se puede sumar directamente la cadena leída.

In [13]:
from io import StringIO
import csv

texto_csv = "aula;temperatura\nA-01;25.5\nB-02;27.6"
archivo = StringIO(texto_csv)
lector_archivo = csv.DictReader(archivo, delimiter=";")

lista_temperatura = []
for fila in lector_archivo:
    lista_temperatura.append(float(fila["temperatura"]))

print(lista_temperatura)
print("Suma de temperaturas", sum(lista_temperatura))


[25.5, 27.6]
Suma de temperaturas 53.1


No se podria sumar directamente dado que se añadaria la cadena al final del numero sin sumarlo, 25.527.6

Ejercicio 09.04 · JSON legible. Crea un diccionario con aula, ocupacion y proyector. Conviértelo a texto JSON con indent y caracteres no ASCII conservados; recupéralo; y comprueba el tipo de cada valor.

In [17]:
import json

diccionario = {"aula": "B-01", "ocupacion": 20, "proyector": True}
texto_json = json.dumps(diccionario, ensure_ascii= False, indent=2)
recuperar_json = json.loads(texto_json)

print(type(recuperar_json["aula"]))
print(type(recuperar_json["ocupacion"]))
print(type(recuperar_json["proyector"]))

<class 'str'>
<class 'int'>
<class 'bool'>


## Proyecto 09 · Informe portable de lecturas

Construye un programa que parta de un texto CSV con tres filas de aula, temperatura y humedad. Debe leerlo con DictReader, convertir los valores, construir una lista de diccionarios de lecturas y generar un informe JSON con las claves total_lecturas, aulas y lecturas. Usa una carpeta temporal durante la práctica para escribir el JSON y leerlo de nuevo.

El proyecto debe comprobar que el JSON recuperado contiene el mismo número de lecturas; mostrar el informe con indent; y documentar en Markdown la ruta relativa que usarías para guardar una entrega real, sin escribir rutas personales. Incluye una tabla de pruebas con un archivo existente, uno ausente, una temperatura que se puede convertir y otra que no. Explica qué dato o mensaje conservarías para investigar una conversión fallida.

No uses pickle ni eval para leer formatos de datos. La entrega debe poder ejecutarse desde un kernel reiniciado y no debe sobrescribir recursos del curso.

In [45]:
import csv
from io import StringIO
import json
from tempfile import TemporaryDirectory
from pathlib import Path


texto_csv = "aula;temperatura;humedad\nA-01;25.5;75\nB-01;19.5;65\nC-01;28.5;45"
archivo = StringIO(texto_csv)
lector = csv.DictReader(archivo, delimiter=";")

dict_tipos_correctos = []
for fila in lector:
    fila_convertida = {
        "aula": fila["aula"],
        "temperatura": float(fila["temperatura"]),
        "humedad": int(fila["humedad"])
    }
    dict_tipos_correctos.append(fila_convertida)

aulas = []
for fila in dict_tipos_correctos:
    aulas.append(fila["aula"])

lecturas = {
    "total_lecturas": len(dict_tipos_correctos),
    "aulas": aulas,
    "lecturas": dict_tipos_correctos
}

numero_lecturas = len(dict_tipos_correctos)
with TemporaryDirectory() as carpeta_temporal:
    ruta = Path(carpeta_temporal)/"archivo.json"
    with open(ruta, mode = "w",encoding="utf-8") as f:
        json.dump(lecturas, indent=2, ensure_ascii=False, fp=f)

    with open(ruta, mode = "r", encoding="utf-8") as f:
        leer_informe = json.load(f)
        print("----INFORME DETALLADO----")
        print(json.dumps(leer_informe,indent=2, ensure_ascii="utf-8"))
        print("Tiene el mismo numero de lecturas? ",leer_informe["total_lecturas"]==numero_lecturas)



----INFORME DETALLADO----
{
  "total_lecturas": 3,
  "aulas": [
    "A-01",
    "B-01",
    "C-01"
  ],
  "lecturas": [
    {
      "aula": "A-01",
      "temperatura": 25.5,
      "humedad": 75
    },
    {
      "aula": "B-01",
      "temperatura": 19.5,
      "humedad": 65
    },
    {
      "aula": "C-01",
      "temperatura": 28.5,
      "humedad": 45
    }
  ]
}
Tiene el mismo numero de lecturas?  True


## Autoevaluación

Comprueba que puedes explicar por qué Path es preferible a construir rutas como texto; qué garantiza with; qué función cumple la codificación UTF-8; por qué CSV necesita un lector específico; qué estructuras de Python representa JSON; y por qué los datos externos deben validarse antes de convertirlos.

## Para ampliar

- [Pathlib](https://docs.python.org/3/library/pathlib.html)
- [Módulo csv](https://docs.python.org/3/library/csv.html)
- [Módulo json](https://docs.python.org/3/library/json.html)
- [Entrada y salida de archivos](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)